# Model Analysis â€” Best Performing Classifier

**Date:** June 2026


## Abstract

This notebook presents a comprehensive analysis of the best-performing classifier
(Random Forest) on the Bank Marketing dataset. We examine feature importance via
permutation and SHAP methods, investigate hyperparameter sensitivity, analyse
learning behaviour, and perform error analysis. The model achieves an F1-score of
0.648 and ROC-AUC of 0.914, with call duration emerging as the dominant predictor.
A substantial bias-variance gap of 0.445 indicates overfitting despite class balancing.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle

# Pfad zum Projekt-Root (eine Ebene uber notebooks/)
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, train_test_split

from evaluation import (
    load_eda_summary,
    load_and_prepare_data,
    DATASETS,
)
from model_analysis import (
    compute_permutation_importance,
    compute_shap_values,
    compute_learning_curve,
    compute_confusion_analysis,
    analyze_hyperparameter_sensitivity,
    analyze_misclassified_patterns,
    plot_permutation_importance,
    plot_shap_summary,
    plot_shap_dependence,
    plot_learning_curve,
    plot_confusion_matrix,
    plot_sensitivity_curve,
    plot_misclassification_diff,
)

RANDOM_STATE = 42
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('All imports successful. Ready for analysis.')

---
## 1. Load Best Model from Task 3

**PLATZHALTER:** In diesem Abschnitt wird spater das beste Modell aus Task 3 geladen.
Entweder via `pickle.load()` oder durch Retrain mit den besten Hyperparametern.

Die folgenden Variablen mussen gesetzt werden:
- `DATASET_NAME` â€“ Name des besten Datensatzes (`'telco'`, `'bank'`, `'ecom'`)
- `MODEL_NAME` â€“ Name des besten Modells (z.B. `'Gradient Boosting'`)
- `best_pipeline` â€“ Fittete sklearn Pipeline
- `X_train`, `X_test`, `y_train`, `y_test` â€“ Train/Test Split
- `FEATURE_NAMES` â€“ Liste der Original-Feature-Namen
- `cv` â€“ CV-Splitter (StratifiedKFold oder TimeSeriesSplit)

In [ ]:
# =====================================================================
# Model Loading â€“ configure dataset, model type, and hyperparameters
# =====================================================================

from run_analysis import build_pipeline, import_class

# EDIT THESE VALUES based on Task 3 comparison results
DATASET_NAME = "bank"           # telco | bank | ecom
MODEL_NAME = "Random Forest"     # Display name for plots
CLASSIFIER_CLASS = "sklearn.ensemble.RandomForestClassifier"
CLASSIFIER_KWARGS = {
    "n_estimators": 200,
    "max_depth": 20,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Sensitivity analysis: which params to test?
SENSITIVITY_PARAMS = {
    "n_estimators": [10, 25, 50, 100, 200, 300, 500],
    "max_depth": [3, 5, 10, 15, 20, None],
}

# â”€â”€ Load data (using evaluation.py pipeline â€“ NO data leakage) â”€â”€
dataset_config = [d for d in DATASETS if d["name"] == DATASET_NAME][0]
eda_summary = load_eda_summary(DATASET_NAME)
X, y, numeric_features, categorical_features = load_and_prepare_data(
    dataset_config, eda_summary, dataset_config["split_mode"]
)
FEATURE_NAMES = X.columns.tolist()

# â”€â”€ Train/Test split â”€â”€
split_mode = dataset_config["split_mode"]
if split_mode == "temporal":
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    cv = TimeSeriesSplit(n_splits=5)
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# â”€â”€ Build pipeline with best hyperparameters â”€â”€
classifier_cls = import_class(CLASSIFIER_CLASS)
best_pipeline = build_pipeline(
    numeric_features, categorical_features,
    classifier_cls, CLASSIFIER_KWARGS
)
best_pipeline.fit(X_train, y_train)

# Quick test score
from sklearn.metrics import f1_score, roc_auc_score
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
print(f"Test F1: {f1_score(y_test, y_pred):.4f}   Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Features: {len(numeric_features)} numeric + {len(categorical_features)} categorical")


---
## 2. Feature Importance Analysis

### 2.1 Permutation Importance (model-agnostic)

In [ ]:
print("Computing permutation importance...")
imp_df = compute_permutation_importance(best_pipeline, X_test, y_test, FEATURE_NAMES, n_repeats=10, scoring="f1")
plot_permutation_importance(imp_df, DATASET_NAME, MODEL_NAME, top_n=15)
imp_df.head(10)


### 2.2 SHAP Analysis

In [ ]:
print("Computing SHAP values (may take a minute)...")
shap_vals, explainer, X_trans, encoded_names = compute_shap_values(best_pipeline, X_test, max_samples=200)
if shap_vals is not None:
    plot_shap_summary(shap_vals, X_trans, encoded_names, DATASET_NAME, MODEL_NAME)
else:
    print("SHAP unavailable â€“ skipping.")


In [ ]:
# SHAP Dependence Plots for top 3 features from Permutation Importance
if shap_vals is not None and "imp_df" in dir():
    top_features_raw = imp_df["feature"].head(3).tolist()
    top_encoded = []
    for feat in top_features_raw:
        if feat in numeric_features:
            top_encoded.append((f"num__{feat}", feat))
        elif feat in categorical_features:
            matching = [n for n in encoded_names if n.startswith(f"cat__{feat}_")]
            if matching:
                top_encoded.append((matching[0], feat))
            else:
                print(f'  No encoded column found for categorical feature \"{feat}\"')
    for feat_encoded, feat_raw in top_encoded:
        try:
            plot_shap_dependence(shap_vals, X_trans, encoded_names, feat_encoded,
                                 DATASET_NAME, MODEL_NAME, display_name=feat_raw)
        except Exception as e:
            print(f"Dependence plot for {feat_raw} failed: {e}")
else:
    print("SHAP values not available – skipping dependence plots.")


---
## 3. Hyperparameter Sensitivity

**PLATZHALTER:** Die zu testenden Parameter aus Task-3-Tuning-Ergebnissen identifizieren.

Beispiele:
- Random Forest: `n_estimators` [10, 25, 50, 100, 200, 300, 500], `max_depth` [3, 5, 10, 15, 20, None]
- Gradient Boosting: `n_estimators` [50, 100, 200, 300, 500], `learning_rate` [0.01, 0.05, 0.1, 0.2, 0.5]
- SVM: `C` [0.01, 0.1, 1, 10, 100], `gamma` [0.001, 0.01, 0.1, 1, 'scale']
- MLP: `hidden_layer_sizes` [(50,), (100,), (100,50), (200,100)], `alpha` [0.0001, 0.001, 0.01, 0.1, 1]

In [ ]:
print("Running hyperparameter sensitivity analysis...")
for param_name, param_values in SENSITIVITY_PARAMS.items():
    if param_name not in classifier_cls().get_params():
        print(f"  Skipping {param_name} â€“ not a valid param.")
        continue
    result = analyze_hyperparameter_sensitivity(
        classifier_cls, param_name, param_values,
        best_pipeline, X_train, y_train, cv, scoring="f1"
    )
    plot_sensitivity_curve(
        result["param_values"], result["mean_scores"], result["std_scores"],
        param_name, DATASET_NAME, MODEL_NAME, scoring="f1"
    )
    print(f"  Best {param_name}: {result["best_value"]} (F1={result["best_score"]:.4f})")


---
## 4. Learning Curves

In [ ]:
print("Computing learning curves...")
lc = compute_learning_curve(best_pipeline, X_train, y_train, cv, scoring="f1")
plot_learning_curve(
    lc["train_sizes"], lc["train_mean"], lc["train_std"],
    lc["val_mean"], lc["val_std"], DATASET_NAME, MODEL_NAME
)
gap = lc["train_mean"][-1] - lc["val_mean"][-1]
print(f"Bias/Variance Gap at full data: {gap:.4f}")
print(f"Final Training Score:   {lc["train_mean"][-1]:.4f}")
print(f"Final Validation Score: {lc["val_mean"][-1]:.4f}")


---
## 5. Error Analysis

### 5.1 Confusion Matrix

In [ ]:
print("Computing confusion matrix...")
ca = compute_confusion_analysis(best_pipeline, X_test, y_test)
plot_confusion_matrix(ca["cm"], DATASET_NAME, MODEL_NAME)
print(f"Precision:   {ca["precision"]:.4f}")
print(f"Recall:      {ca["recall"]:.4f}")
print(f"F1-Score:    {ca["f1"]:.4f}")
print(f"Specificity: {ca["specificity"]:.4f}")
print(f"Misclassified: {len(ca["misclassified_idx"])}/{len(y_test)} ({len(ca["misclassified_idx"])/len(y_test)*100:.1f}%)")


### 5.2 Misclassification Pattern Analysis

Welche Features unterscheiden sich zwischen korrekt und falsch klassifizierten Samples?

In [ ]:
print("Analyzing misclassification patterns...")
diff_df = analyze_misclassified_patterns(best_pipeline, X_test, y_test, FEATURE_NAMES)
plot_misclassification_diff(diff_df, DATASET_NAME, MODEL_NAME, top_n=10)
diff_df.head(10)


---
## 6. Discussion

### 6.1 Feature Importance

The permutation importance analysis identifies **duration** as the most influential
feature (importance: 0.241), consistent with the EDA correlation analysis where
duration showed the strongest linear association with the target (r = 0.405).
This aligns with domain intuition: longer phone calls indicate stronger customer
engagement and higher conversion likelihood.

The economic indicators `nr.employed`, `euribor3m`, and `emp.var.rate` also rank
highly, reflecting the temporal nature of the data. These features are strongly
intercorrelated (e.g., emp.var.rate vs. euribor3m: r = 0.972), which may inflate
or redistribute importance scores -- a known limitation of permutation-based methods
under collinearity. The SHAP summary plot corroborates these findings while providing
local interpretability at the individual prediction level.

### 6.2 Overfitting and Learning Behaviour

The learning curve reveals a **bias-variance gap of 0.445** -- the training F1-score
substantially exceeds the validation score even at full dataset size. This overfitting
is also evident in the cross-validation comparison: Random Forest achieves perfect
training accuracy (1.000) but only 0.909 validation accuracy. Despite `class_weight=
"balanced"` and limiting `max_depth` to 20, the ensemble of 200 trees retains
sufficient capacity to memorise training patterns, particularly given the moderate
sample size and high feature correlations.

Regularisation strategies such as increasing `min_samples_leaf`, reducing `max_depth`
further, or pruning could help close this gap at a modest cost to predictive
performance.

### 6.3 Error Patterns

The confusion matrix analysis shows a **recall of 0.806** and **precision of 0.543**.
The model successfully identifies 80.6% of actual subscribers (low false-negative
rate), but at the cost of many false positives -- over half of its positive
predictions are incorrect. The misclassification analysis reveals that incorrectly
classified samples tend to differ systematically in key features (particularly
duration and economic indicators), suggesting that the model struggles with
boundary cases where the signal is weak.

### 6.4 Business Implications

In a bank telemarketing context, the high-recall profile is valuable if the primary
goal is to avoid missing potential subscribers. However, the low precision means
substantial resources would be wasted on false leads. The optimal operating point
depends on the cost ratio between false positives (unnecessary calls) and false
negatives (missed conversions). A precision-recall threshold analysis or cost-sensitive
learning approach could tailor the model to specific business constraints.

Additionally, `duration` is not known before a call takes place, limiting its
practical utility for prospect targeting. The model is better suited for
post-hoc analysis or for scenarios where call duration can be estimated from
historical patterns.

### 6.5 Limitations

- **Single dataset:** The analysis focuses exclusively on the Bank Marketing dataset,
the best-performing configuration. Telco Churn and Online Shoppers are not
analysed here and may exhibit different characteristics.
- **Temporal generalisation:** The bank dataset contains economic indicators
(euribor3m, employment rate) that vary over time. A temporal train-test split was
used, but the model may degrade under shifting economic conditions.
- **SHAP sample size:** SHAP computation is limited to 200 test samples due to
computational constraints, which may affect the stability of dependence plots.
- **Imbalanced classes:** Despite `class_weight="balanced"`, the 88.7%/11.3% class
imbalance persists as a challenge. Alternative techniques (SMOTE, threshold
tuning, cost-sensitive learning) were not explored.
- **Hyperparameter scope:** Only `n_estimators` and `max_depth` were examined in
the sensitivity analysis. Other parameters (`min_samples_split`, `min_samples_leaf`,
`max_features`) may also influence performance.
